# 🔥 Advanced · Fine-tune a VLM

> 🔥 **Advanced / heavy lab.** Adapt a vision-language model (SmolVLM) to answer questions about images.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **30–60 min** including downloads. Built on the official **[TRL + SmolVLM](https://huggingface.co/docs/trl/en/sft_trainer#vision-language-models)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

A VLM = a vision encoder feeding image tokens into an LLM (track-C/D themes: grounding language in pixels). Here we LoRA-fine-tune one on a VQA dataset.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — SmolVLM 2B, 4-bit | A100 40 GB — Qwen2-VL-7B / PaliGemma |
| **Storage** | ChartQA subset ~ 1 GB + model | VQA datasets 10–100 GB; 50–150 GB disk |
| **Time** | 200 steps ~ 30–60 min | full dataset ~ hours–1 day |

**Full pipeline (scale-up):** `accelerate launch --multi_gpu vlm_sft.py …`; larger backbone, full VQA.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

## 1 · Load SmolVLM + processor (4-bit)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
ckpt = "HuggingFaceTB/SmolVLM-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained(ckpt)
model = AutoModelForVision2Seq.from_pretrained(ckpt, quantization_config=bnb, device_map="auto")

## 2 · A small image-QA dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("HuggingFaceM4/ChartQA", split="train[:500]")
print(ds, "\n", ds[0].keys())

## 3 · Collator — build chat messages with the image, then train

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
def collate(examples):
    texts, images = [], []
    for ex in examples:
        msg = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": ex["query"]}]},
               {"role": "assistant", "content": [{"type": "text", "text": ex["label"][0]}]}]
        texts.append(processor.apply_chat_template(msg, tokenize=False))
        images.append([ex["image"]])
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone(); labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch
args = SFTConfig(output_dir="vlm-out", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                 max_steps=200, learning_rate=2e-4, logging_steps=20, bf16=True,
                 remove_unused_columns=False, dataset_kwargs={"skip_prepare_dataset": True}, report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, data_collator=collate,
                     peft_config=LoraConfig(r=8, lora_alpha=16, task_type="CAUSAL_LM", target_modules="all-linear"))
trainer.train()

## 4 · Ask the fine-tuned model about a chart

In [ ]:
msg = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What is the highest value in this chart?"}]}]
prompt = processor.apply_chat_template(msg, add_generation_prompt=True)
inp = processor(text=prompt, images=[ds[0]["image"]], return_tensors="pt").to(model.device)
out = model.generate(**inp, max_new_tokens=64)
print(processor.decode(out[0], skip_special_tokens=True))

## Evaluate — held-out loss (lower = better)
Run on a ChartQA slice held out from training; for task accuracy, generate answers and exact-match against the gold labels.

In [ ]:
from datasets import load_dataset
ev = load_dataset("HuggingFaceM4/ChartQA", split="train[500:600]")
print(trainer.evaluate(eval_dataset=ev))

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Grounding language in pixels feeds two tracks:
- **C · Egocentric** first-person visual question answering · **D · Scene / world** open-vocabulary scene understanding (OpenScene / LERF use vision-language features).

## Troubleshooting & next steps
- VLM collators are model-specific — SmolVLM/Idefics3, PaliGemma, Qwen2-VL each differ; follow the matching TRL example.
- Image-token handling changes across `transformers` versions; pin if generation misaligns.
- Bigger options: **Qwen2-VL-2B**, **PaliGemma-3B**, **LLaVA-1.6**.